In [1]:
# Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode
from pyspark.sql.functions import col, count
from pyspark.sql.functions import input_file_name
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col

In [2]:
# Configuração do Spark com Delta Lake
builder = SparkSession.builder \
    .appName("ProjetoMedico") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.debug.maxToStringFields", "5000") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.files.maxPartitionBytes", "134217728") \
    .config("spark.driver.extraClassPath", "/repositorio/postgresql-42.6.2.jar") \
    .config("spark.jars", "/repositorio/postgresql-42.6.2.jar")


<h1>Lendo os arquivos Json com Spark e salvando particionado em Delta Lake</h1>

In [3]:
# Cria a sessão
spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-46dbfc5d-01e7-4897-94c5-015c4fe9c8f6;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 155ms :: artifacts dl 8ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

In [4]:
# Caminho da tabela Delta
delta_table_path = "/repositorio/raw/delta-lake"

In [5]:
# Caminho dos arquivos JSON
json_path = "/repositorio/data"

In [6]:
# Carregar os arquivos JSON em um DataFrame do Spark
df = spark.read.option("multiline", "true").json(json_path)

In [7]:
# Adicionar coluna com nome do arquivo de origem
df = df.withColumn("arquivo_origem", input_file_name())

df.show(5, truncate=False)

IOPub data rate exceeded.                                                       
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [8]:
# Escrever no formato Delta, particionado pelo arquivo de origem
df.write.format("delta").mode("overwrite").partitionBy("arquivo_origem").save(delta_table_path)

<h1>Lendo e tratando os arquivos em delta com Spark </h1>

In [9]:
# Ler os dados no formato Delta
df_final = spark.read.format("delta").load(delta_table_path)

df_final.show(5, truncate=False)

IOPub data rate exceeded.                                                       
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [10]:
# Explodir a coluna entry para explorar cada recurso individualmente
df_explode = df.withColumn("entry", explode(df["entry"]))

# Criar um DataFrame com os recursos
df_resource = df_explode.select("entry.resource.*")

<h1>As condições médicas</h1>

In [11]:
# Filtrar apenas os recursos do tipo "Condition"
df_condition = df_resource.filter(col("resourceType") == "Condition")

# Selecionar apenas a coluna "code.text"
df_condition_code_text = df_condition.select(col("code.text").alias("condicao"))

# Contar e ordenar as condições mais comuns
df_top_condition = (
    df_condition_code_text
    .groupBy("condicao")
    .agg(count("*").alias("n_ocorrencias"))
    .orderBy(col("n_ocorrencias").desc())
)

df_top_condition.show(10, truncate=False)


[Stage 38:=====================================================>  (38 + 2) / 40]

+---------------------------------------+-------------+
|condicao                               |n_ocorrencias|
+---------------------------------------+-------------+
|Viral sinusitis (disorder)             |1031         |
|Acute viral pharyngitis (disorder)     |641          |
|Acute bronchitis (disorder)            |454          |
|Normal pregnancy                       |426          |
|Prediabetes                            |340          |
|Anemia (disorder)                      |331          |
|Body mass index 30+ - obesity (finding)|290          |
|Hypertension                           |283          |
|Chronic sinusitis (disorder)           |212          |
|Miscarriage in first trimester         |170          |
+---------------------------------------+-------------+
only showing top 10 rows



In [12]:
df_top_condition.printSchema()

root
 |-- condicao: string (nullable = true)
 |-- n_ocorrencias: long (nullable = false)



<h1>Os medicamentos prescritos</h1>

In [13]:
# Filtrar pelo recurso "MedicationRequest"
df_medication = df_resource.filter(col("resourceType") == "MedicationRequest")

# Selecionar apenas a coluna "medicationCodeableConcept.text"
df_medication_text = df_medication.select(col("medicationCodeableConcept.text").alias("medicacao"))

# Contar e ordenar os medicamentos mais prescritos
df_top_medication = (
    df_medication_text
    .groupBy("medicacao")
    .agg(count("*").alias("n_prescricoes"))
    .orderBy(col("n_prescricoes").desc())
)

df_top_medication.show(10, truncate=False)


[Stage 41:======================================================> (39 + 1) / 40]

+---------------------------------------------------+-------------+
|medicacao                                          |n_prescricoes|
+---------------------------------------------------+-------------+
|1 ML Epoetin Alfa 4000 UNT/ML Injection [Epogen]   |2919         |
|Simvistatin 10 MG                                  |2000         |
|Cisplatin 50 MG Injection                          |691          |
|PACLitaxel 100 MG Injection                        |631          |
|Acetaminophen 325 MG Oral Tablet                   |507          |
|Naproxen sodium 220 MG Oral Tablet                 |228          |
|Amoxicillin 250 MG / Clavulanate 125 MG Oral Tablet|219          |
|Ibuprofen 200 MG Oral Tablet                       |147          |
|Hydrochlorothiazide 25 MG                          |123          |
|Acetaminophen 160 MG Oral Tablet                   |118          |
+---------------------------------------------------+-------------+
only showing top 10 rows



In [14]:
df_top_medication.printSchema()

root
 |-- medicacao: string (nullable = true)
 |-- n_prescricoes: long (nullable = false)



<h1>Gênero dos pacientes</h1>

In [15]:
# Filtrar pelo recurso "Patient"
df_patient = df_resource.filter(col("resourceType") == "Patient")

# Selecionar apenas a coluna "gender"
df_patient_gender = df_patient.select(col("gender").alias("sexo"))

# Contar pacientes do sexo masculino
df_gender_count = (
    df_patient_gender
    .groupBy(col("sexo"))
    .agg(count("*").alias("qtd"))
    .orderBy(col("qtd").desc())
)

df_gender_count.show()

[Stage 44:=====================================================>  (38 + 2) / 40]

+------+---+
|  sexo|qtd|
+------+---+
|female|509|
|  male|491|
+------+---+



In [17]:
df_gender_count.printSchema()

root
 |-- sexo: string (nullable = true)
 |-- qtd: long (nullable = false)



<h1>Configurando o Bando de Dados Postgres</h1>

In [18]:
!pip install psycopg2-binary

import psycopg2

# Defina as configurações do PostgreSQL
jdbc_url = "jdbc:postgresql://postgres:5432/deltalake-db"
properties = {
    "user": "admin",
    "password": "admin",
    "driver": "org.postgresql.Driver"
}

# Tente conectar ao PostgreSQL
try:
    conn = psycopg2.connect(
        host="postgres", 
        port="5432", 
        database="deltalake-db", 
        user="admin", 
        password="admin"
    )
    print("Conexão bem-sucedida ao PostgreSQL!")

    # Realizar uma consulta simples para verificar se a conexão está funcionando
    cur = conn.cursor()
    cur.execute("SELECT 1;")
    result = cur.fetchone()
    if result:
        print("Consulta bem-sucedida!")
    else:
        print("Erro na consulta!")

except Exception as e:
    print(f"Erro ao conectar ao PostgreSQL: {e}")


Conexão bem-sucedida ao PostgreSQL!
Consulta bem-sucedida!


In [19]:
# Comando SQL para criar o schema
create_schema_sql = """
CREATE SCHEMA IF NOT EXISTS projeto_medico;
"""

# Executar o comando SQL para criar o schema
cur.execute(create_schema_sql)

# Commit para salvar as alterações no banco
conn.commit()

In [20]:
# Nome da tabela 
table_name = "projeto_medico.conditions"

# Criar conexão JDBC para execução de comando DDL
conn = spark._jvm.java.sql.DriverManager.getConnection(jdbc_url, properties["user"], properties["password"])
stmt = conn.createStatement()

# Verificar se a tabela tem registros antes do TRUNCATE
count_query = f"SELECT COUNT(*) FROM {table_name};"
count_result = stmt.executeQuery(count_query)
count_result.next()
row_count = count_result.getInt(1)

if row_count > 0:
    print(f"A tabela {table_name} tem {row_count} registros. Executando TRUNCATE...")
    stmt.executeUpdate(f"TRUNCATE TABLE {table_name};")
    print(f"Tabela {table_name} truncada com sucesso!")
else:
    print(f"A tabela {table_name} já está vazia.")

A tabela projeto_medico.conditions tem 136 registros. Executando TRUNCATE...
Tabela projeto_medico.conditions truncada com sucesso!


In [21]:
# Nome da tabela
table_name2 = "projeto_medico.medications"

# Criar conexão JDBC para execução de comando DDL
conn = spark._jvm.java.sql.DriverManager.getConnection(jdbc_url, properties["user"], properties["password"])
stmt = conn.createStatement()

# Verificar se a tabela tem registros antes do TRUNCATE
count_query = f"SELECT COUNT(*) FROM {table_name2};"
count_result = stmt.executeQuery(count_query)
count_result.next()
row_count = count_result.getInt(1)

if row_count > 0:
    print(f"A tabela {table_name2} tem {row_count} registros. Executando TRUNCATE...")
    stmt.executeUpdate(f"TRUNCATE TABLE {table_name2};")
    print(f"Tabela {table_name2} truncada com sucesso!")
else:
    print(f"A tabela {table_name2} já está vazia.")

A tabela projeto_medico.medications tem 148 registros. Executando TRUNCATE...
Tabela projeto_medico.medications truncada com sucesso!


In [22]:
# Nome da tabela
table_name3 = "projeto_medico.gender_patient"

# Criar conexão JDBC para execução de comando DDL
conn = spark._jvm.java.sql.DriverManager.getConnection(jdbc_url, properties["user"], properties["password"])
stmt = conn.createStatement()

# Verificar se a tabela tem registros antes do TRUNCATE
count_query = f"SELECT COUNT(*) FROM {table_name3};"
count_result = stmt.executeQuery(count_query)
count_result.next()
row_count = count_result.getInt(1)

if row_count > 0:
    print(f"A tabela {table_name3} tem {row_count} registros. Executando TRUNCATE...")
    stmt.executeUpdate(f"TRUNCATE TABLE {table_name3};")
    print(f"Tabela {table_name3} truncada com sucesso!")
else:
    print(f"A tabela {table_name3} já está vazia.")

A tabela projeto_medico.gender_patient já está vazia.


<h1>Inserindo os dados no Banco de Dados PostgreSQL</h1>

In [23]:
# Escrever o primeiro DataFrame na tabela "conditions"
df_top_condition.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/deltalake-db") \
    .option("dbtable", "projeto_medico.conditions") \
    .option("user", "admin") \
    .option("password", "admin") \
    .mode("overwrite") \
    .save()

# Escrever o segundo DataFrame na tabela "medications"
df_top_medication.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/deltalake-db") \
    .option("dbtable", "projeto_medico.medications") \
    .option("user", "admin") \
    .option("password", "admin") \
    .mode("overwrite") \
    .save()

# Escrever o terceiro DataFrame na tabela "gender_patient"
df_gender_count.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/deltalake-db") \
    .option("dbtable", "projeto_medico.gender_patient") \
    .option("user", "admin") \
    .option("password", "admin") \
    .mode("overwrite") \
    .save()

print("DataFrames escritos com sucesso!")


[Stage 63:=====================================================>  (38 + 2) / 40]

DataFrames escritos com sucesso!


<h1>Consultando os dados nas tabelas dentro do Banco de Dados</h1>

In [24]:
# Consultando dados de uma tabela 
df_conditions = spark.read.jdbc(url=jdbc_url, table="projeto_medico.conditions", properties=properties)

# Exibindo o DataFrame
df_conditions.show(10, truncate=False)

+---------------------------------------+-------------+
|condicao                               |n_ocorrencias|
+---------------------------------------+-------------+
|Viral sinusitis (disorder)             |1031         |
|Acute viral pharyngitis (disorder)     |641          |
|Acute bronchitis (disorder)            |454          |
|Normal pregnancy                       |426          |
|Prediabetes                            |340          |
|Anemia (disorder)                      |331          |
|Body mass index 30+ - obesity (finding)|290          |
|Hypertension                           |283          |
|Chronic sinusitis (disorder)           |212          |
|Miscarriage in first trimester         |170          |
+---------------------------------------+-------------+
only showing top 10 rows



In [25]:
# Consultando dados de uma tabela 
df_medication = spark.read.jdbc(url=jdbc_url, table="projeto_medico.medications", properties=properties)

# Exibindo o DataFrame
df_medication.show(10, truncate=False)

+---------------------------------------------------+-------------+
|medicacao                                          |n_prescricoes|
+---------------------------------------------------+-------------+
|1 ML Epoetin Alfa 4000 UNT/ML Injection [Epogen]   |2919         |
|Simvistatin 10 MG                                  |2000         |
|Cisplatin 50 MG Injection                          |691          |
|PACLitaxel 100 MG Injection                        |631          |
|Acetaminophen 325 MG Oral Tablet                   |507          |
|Naproxen sodium 220 MG Oral Tablet                 |228          |
|Amoxicillin 250 MG / Clavulanate 125 MG Oral Tablet|219          |
|Ibuprofen 200 MG Oral Tablet                       |147          |
|Hydrochlorothiazide 25 MG                          |123          |
|Acetaminophen 160 MG Oral Tablet                   |118          |
+---------------------------------------------------+-------------+
only showing top 10 rows



In [26]:
# Consultando dados de uma tabela 
df_gender = spark.read.jdbc(url=jdbc_url, table="projeto_medico.gender_patient", properties=properties)

# Exibindo o DataFrame
df_gender.show(10, truncate=False)

+------+---+
|sexo  |qtd|
+------+---+
|female|509|
|male  |491|
+------+---+

